# Fetch Opik experiment results (metric review)

Pull experiment rows from Opik: **dataset context**, **grader output**, and **metric scores + reasons** for LLM or manual metric review.

1. Run **bootstrap** → **config** (use `RUN_PRESET` for your eval).
2. **List experiments** (optional) → copy `EXPERIMENT_ID` from the table (not the project id from the browser URL).
3. **Load** → **summary** → **duration check** → **filter** → **preview** / **export**.

**Textual baseline (screenshot):** `RUN_PRESET = "textual_baseline_gpt_chat_latest"`, project `AI Evaluations Baseline V2`, metrics include `grader_duration_sec` / `judge_duration_sec`.

Optional: if `eval_results` exists in this kernel from `evaluate()`, use **from in-memory eval_results**.

In [ ]:
import sys
from pathlib import Path

_repo = Path.cwd()
if not (_repo / "notebook_env.py").exists():
    _fallback = Path("/home/nxtwave/Repositories/opik-ai-evaluations")
    _repo = _fallback if (_fallback / "notebook_env.py").exists() else _repo
sys.path.insert(0, str(_repo))

from notebook_env import activate

REPO_ROOT = activate()

import opik
import pandas as pd
from IPython.display import display

print("OK —", sys.executable)
print("repo:", REPO_ROOT)
print("opik", opik.__version__, "| pandas", pd.__version__)

## Config

In [ ]:
import importlib
import os
from datetime import datetime

import notebook_env
from opik import Opik

import experiment_review

importlib.reload(notebook_env)
importlib.reload(experiment_review)

notebook_env.load_env_file(override=True)

os.environ.setdefault("OPIK_URL_OVERRIDE", "https://3.110.54.210/api")
os.environ.setdefault("OPIK_CHECK_TLS_CERTIFICATE", "false")
OPIK_UI_HOST = "https://3.110.54.210"

os.environ.setdefault("OPIK_PROJECT_NAME", "AI Evaluations Baseline V2")

# --- Run preset (fills project, surface, experiment name, dataset) ---
RUN_PRESET = "textual_baseline_gpt_chat_latest"
run = experiment_review.apply_run_preset(RUN_PRESET)

PROJECT_NAME = run.get("project_name") or os.environ["OPIK_PROJECT_NAME"]
os.environ["OPIK_PROJECT_NAME"] = PROJECT_NAME
opik_client = Opik(project_name=PROJECT_NAME)

SURFACE = run["surface"]
DATASET_NAME = run.get("dataset_name")
EXPERIMENT_NAME = run.get("experiment_name")
USE_LATEST_RUN = bool(run.get("use_latest_run", True))

# Override preset: set experiment id from **list experiments** table (not project URL)
EXPERIMENT_ID = run.get("experiment_id")  # e.g. "01a13fa3-22bd-76e2-a271-c048836e7756"

NAME_SEARCH = run.get("name_search")  # set None to skip listing

TRUNCATE_ITEMS = False
ENRICH_DATASET = True
ATTACH_TRACE_DURATION = True
TRACE_DURATION_MAX_ROWS = 50  # None = all rows (slower)
ONLY_FAILS = False
MAX_ROWS = None
METRIC_NAMES = None

surface_preset = experiment_review.SURFACE_PRESETS[SURFACE]
CONTEXT_KEYS = surface_preset["context_keys"]
METRICS = METRIC_NAMES or surface_preset["metrics"]

EXPORT_DIR = REPO_ROOT / "data" / "exports" / "reviews"

print("run preset:", RUN_PRESET)
print("project:", PROJECT_NAME)
print("surface:", SURFACE)
print("dataset:", DATASET_NAME)
print("experiment_name:", EXPERIMENT_NAME)
print("experiment_id:", EXPERIMENT_ID or "(auto: latest run)")
print("metrics:", METRICS)

In [ ]:
if NAME_SEARCH:
    matches = experiment_review.list_experiments_by_name_substring(
        opik_client, NAME_SEARCH, project_name=PROJECT_NAME
    )
    display(pd.DataFrame(matches, columns=["id", "name", "dataset_name"]))
else:
    print("Set NAME_SEARCH to list experiments, or set EXPERIMENT_ID directly.")

## Load from Opik API

In [ ]:
experiment = experiment_review.resolve_experiment(
    opik_client,
    experiment_id=EXPERIMENT_ID,
    experiment_name=EXPERIMENT_NAME,
    project_name=PROJECT_NAME,
    use_latest_run=USE_LATEST_RUN,
)

urls = experiment_review.experiment_dashboard_urls(
    OPIK_UI_HOST, experiment.id, PROJECT_NAME
)
print(f"name: {experiment.name}")
print(f"id:   {experiment.id}")
print(f"dataset: {experiment.dataset_name}")
print("compare:", urls["experiment"])

df = experiment_review.load_experiment_review_dataframe(
    opik_client,
    experiment,
    truncate=TRUNCATE_ITEMS,
    enrich_dataset=ENRICH_DATASET,
    project_name=PROJECT_NAME,
)
if ATTACH_TRACE_DURATION:
    df = experiment_review.attach_trace_durations(
        opik_client, df, max_rows=TRACE_DURATION_MAX_ROWS
    )

print(f"rows: {len(df)}")
metric_cols = [c for c in df.columns if c.startswith("metric__")]
print("metric columns:", [c.replace("metric__", "") for c in metric_cols])
df.head()

## Or load from in-memory `eval_results` (same kernel as `evaluate()`)

In [ ]:
try:
    eval_results  # noqa: F821 — defined after running an eval notebook cell
    df = experiment_review.evaluation_result_to_dataframe(eval_results)
    print(f"Loaded {len(df)} rows from eval_results in memory")
    display(df.head())
except NameError:
    print("No eval_results in this kernel — use API load cell above.")

## Summary & filter

In [ ]:
summary = experiment_review.metric_summary(df)
display(summary)

review_df = experiment_review.filter_review_dataframe(
    df,
    metric_names=METRICS,
    only_fails=ONLY_FAILS,
    max_rows=MAX_ROWS,
)
print(f"review rows: {len(review_df)}")

flat_cols = [
    c
    for c in review_df.columns
    if not c.endswith("_json") and c != "grader_output"
]
display(review_df[flat_cols].head(10))

## Duration check (grader vs judge vs trace)

In [ ]:
try:
    duration_stats = experiment_review.duration_sanity_check(df)
    display(duration_stats)
    print(
        "trace_duration ≈ full row time in UI; "
        "grader_plus_judge ≈ API time for grader + judge LLM calls."
    )
except ValueError as exc:
    print(exc)

## Preview one row (LLM / manual metric check)

In [ ]:
ROW_INDEX = 0

if len(review_df) == 0:
    print("No rows in review_df.")
else:
    packet = experiment_review.build_metric_review_packet(
        review_df.iloc[ROW_INDEX].to_dict(),
        context_keys=CONTEXT_KEYS,
        metrics=METRICS,
    )
    md = experiment_review.packet_to_markdown(packet)
    print(md)

## Optional: trace details for one row

In [ ]:
if len(review_df) > 0 and review_df.iloc[ROW_INDEX].get("trace_id"):
    trace_id = review_df.iloc[ROW_INDEX]["trace_id"]
    trace = opik_client.get_trace_content(id=trace_id)
    print("trace_id:", trace.id)
    for attr in ("name", "duration", "input", "output"):
        if hasattr(trace, attr):
            val = getattr(trace, attr)
            if isinstance(val, str) and len(val) > 2000:
                val = val[:2000] + "..."
            print(f"{attr}:", val)
else:
    print("No trace_id for selected row.")

## Export JSONL for batch review

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
try:
    exp_id = experiment.id[:8]
except NameError:
    exp_id = "memory"
stem = f"{SURFACE}_{exp_id}_{stamp}"

full_path = EXPORT_DIR / f"{stem}_full.jsonl"
review_path = EXPORT_DIR / f"{stem}_metric_review.jsonl"

export_df = review_df.copy()
for col in ("task_output_json", "dataset_item_json"):
    if col in export_df.columns:
        export_df[col] = export_df[col].apply(
            lambda x: x if isinstance(x, (dict, list)) else x
        )

export_df.to_json(full_path, orient="records", lines=True, force_ascii=False)
n = experiment_review.export_review_jsonl(
    review_df,
    review_path,
    context_keys=CONTEXT_KEYS,
    metrics=METRICS,
)

print("full:", full_path)
print("review packets:", review_path, f"({n} rows)")